[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/19.%20The%20SLAP%20for%20Reefer%20Containers%20Problem/P19-Tier-5_executed.ipynb)

---


# 19. The SLAP for Reefer Containers Problem
## Tier 5: Integrated Digital Twin

### Purpose and scope

This notebook builds a small, discrete-time digital-twin simulation for a reefer-container terminal. It represents containers, storage locations, sensors, temperature and power states, alerts, simple risk indicators, and disruption scenarios. The simulation can help readers examine how an externally supplied container-to-location assignment behaves over time.

The assignment is **not optimized in this notebook**. The `assignment_costs` data are retained as terminal input, but the simulation does not use them to choose locations. This distinction matters: the digital twin monitors and simulates an assignment; it does not, by itself, solve the storage location assignment problem (SLAP).

### Digital-twin layers

The model follows a four-layer view:

1. **Physical layer:** containers, storage locations, power demand, and container temperatures.
2. **Connectivity layer:** simulated IoT readings with timestamps and sensor-health fields.
3. **Data-processing layer:** alerts, key performance indicators (KPIs), and heuristic risk indicators.
4. **Application layer:** scenario summaries and visualizations for decision support.

A digital twin is a model connected to observations of a physical system. Here, both the physical process and sensor readings are simulated; there is no live terminal connection, communication network, or external data feed.

### Model assumptions and limitations

- Each container is assigned to one location for the duration of a scenario. The supplied example assignment is illustrative and may exceed a location's power capacity; overloads are reported rather than prevented.
- A location's temperature target is represented by the midpoint of its stated temperature range. Container temperatures move toward that target according to a simple discrete-time thermal model.
- Power demand is the sum of assigned containers' stated kW requirements. The model does not simulate voltage, power quality, refrigeration equipment, or energy consumption over time in kWh.
- Sensor readings include random noise. Sensor accuracy affects the noise scale, but sensors do not fail to communicate and readings are not used to estimate hidden physical states.
- The risk indicators are transparent heuristics, not trained forecasting models. No accuracy, calibration, or predictive validation is claimed.
- Cargo integrity is measured as the percentage of assigned containers within their stated temperature tolerance; it is not weighted by cargo value.
- Emergency actions use simple rules and available backup-location power capacity. They do not model travel time, handling capacity, generator dispatch, or the detailed electrical consequences of an outage.

### Simulation logic

For container \(c\) assigned to location \(l\), one temperature update is

$$
T_{c,t+1} = T_{c,t} + \alpha_c (T_l - T_{c,t})\Delta t + \varepsilon_{c,t},
\qquad
\alpha_c = 0.1\,m_c q_c,
\qquad
\varepsilon_{c,t} \sim \mathcal{N}(0,\,0.1^2\Delta t).
$$

Here, \(T_{c,t}\) is the container temperature at time \(t\), \(T_l\) is the midpoint temperature of its assigned location, \(m_c\) is the container's thermal-mass coefficient, \(q_c\) is its insulation-quality coefficient, and Δt is the time-step duration in hours. The simulation implements this update in `_update_container_temperatures`. It is a simplified first-order model, not a detailed heat-transfer calculation.

The location power load is \(P_l = \sum_{c\ assigned\ to\ l} p_c\), where \(p_c\) is the container's power demand. The code compares \(P_l\) with the location capacity to identify overloads. A scenario can therefore expose an infeasible assignment without changing or optimizing it.

For each time step, the simulation records readings, updates physical state, checks alerts, calculates KPIs, and computes heuristic risk indicators. Results should be interpreted as outputs of these stated assumptions, not as live measurements or validated operational forecasts.

In [ ]:
from collections import deque
from dataclasses import dataclass, replace
from datetime import datetime, timedelta
import random
import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np

print("Imports are ready for the reefer-terminal simulation.")

In [ ]:
@dataclass
class ReeferContainer:
    """Operating and cargo attributes for one reefer container."""

    id: int
    energy: float  # Required electrical power in kW
    temp_req: float  # Required cargo temperature in degrees Celsius
    temp_tolerance: float  # Permitted deviation in degrees Celsius
    dwell_time: float  # Planned storage duration in hours
    cargo_type: str
    cargo_value: float  # Cargo value in thousands of dollars
    thermal_mass: float  # Dimensionless coefficient used by the thermal model
    insulation_quality: float  # Dimensionless coefficient in the range [0, 1]


@dataclass
class StorageLocation:
    """Temperature range, power capacity, and infrastructure of a location."""

    id: int
    power_cap: float  # Available power capacity in kW
    temp_min: float  # Minimum supported temperature in degrees Celsius
    temp_max: float  # Maximum supported temperature in degrees Celsius
    base_cost: float  # Assignment-cost input; not optimized in this notebook
    backup_power: bool
    sensor_count: int  # Infrastructure descriptor, not the simulated sensor count
    location_type: str


@dataclass
class IoT_Sensor:
    """A simulated sensor associated with a location or a container."""

    id: str
    sensor_type: str
    location_id: Optional[int]
    container_id: Optional[int]
    accuracy: float  # Nominal accuracy in the range [0, 1]
    sampling_rate: float  # Samples per second
    battery_level: float  # Battery fraction in the range [0, 1]
    last_maintenance: datetime

print("The container, location, and sensor data structures are defined.")

## 1. Define the terminal instance

The following data describe the demonstration instance. Container power demand is measured in kW; temperatures are in degrees Celsius; dwell time is in hours; and cargo value is expressed in thousands of dollars. The assignment-cost table is part of the input data, but the simulation below does not use it to select an assignment.

In [ ]:
containers = [
    ReeferContainer(1, 45, -18, 2.0, 96, "Pharmaceutical", 500.0, 0.8, 0.9),
    ReeferContainer(2, 65, 3, 1.5, 72, "Fresh Produce", 80.0, 0.6, 0.7),
    ReeferContainer(3, 50, -20, 1.0, 120, "Frozen Seafood", 120.0, 0.9, 0.85),
    ReeferContainer(4, 35, 5, 2.5, 48, "Dairy Products", 60.0, 0.7, 0.8),
    ReeferContainer(5, 55, -15, 1.8, 84, "Chemicals", 200.0, 0.75, 0.95),
    ReeferContainer(6, 40, 2, 2.0, 60, "Beverages", 40.0, 0.65, 0.75),
    ReeferContainer(7, 60, -22, 0.8, 108, "Vaccines", 800.0, 0.95, 0.98),
    ReeferContainer(8, 48, -12, 2.2, 96, "Ice Cream", 90.0, 0.85, 0.88),
]

locations = [
    StorageLocation(1, 120, -30, -5, 15, True, 15, "Ultra-Cold"),
    StorageLocation(2, 100, -25, 8, 12, True, 12, "Cold"),
    StorageLocation(3, 85, -20, 10, 10, False, 10, "Cool"),
    StorageLocation(4, 90, -15, 12, 11, True, 11, "General"),
]

assignment_costs = {
    (1, 1): 18, (1, 2): 20, (1, 3): 25, (1, 4): 22,
    (2, 1): 22, (2, 2): 12, (2, 3): 18, (2, 4): 16,
    (3, 1): 20, (3, 2): 30, (3, 3): 28, (3, 4): 32,
    (4, 1): 15, (4, 2): 10, (4, 3): 14, (4, 4): 11,
    (5, 1): 25, (5, 2): 16, (5, 3): 20, (5, 4): 18,
    (6, 1): 16, (6, 2): 13, (6, 3): 15, (6, 4): 12,
    (7, 1): 30, (7, 2): 35, (7, 3): 40, (7, 4): 38,
    (8, 1): 20, (8, 2): 18, (8, 3): 22, (8, 4): 19,
}

print(f"Demonstration instance: {len(containers)} containers and {len(locations)} locations")
print("\nContainer data:")
for container in containers:
    print(
        f"  C{container.id}: {container.cargo_type}, {container.energy} kW, "
        f"{container.temp_req} ± {container.temp_tolerance} °C, "
        f"${container.cargo_value:.1f}K, dwell {container.dwell_time} h"
    )

print("\nLocation data:")
for location in locations:
    backup_status = "yes" if location.backup_power else "no"
    print(
        f"  L{location.id}: {location.power_cap} kW, "
        f"[{location.temp_min}, {location.temp_max}] °C, "
        f"{location.location_type}, backup power: {backup_status}"
    )

## 2. Build the digital twin

A **state** is the collection of values that describe the simulated system at a particular time. The twin stores container assignments, temperatures, location power loads, grid status, sensor readings, alerts, and KPI history. Each call to `simulate_step` advances this state by one time interval.

The discrete-time loop is the simulation's central mechanism: it converts the current state into readings and updated state, then derives alerts and summaries from those values. This is a simulation workflow, not an optimization algorithm or a learned forecasting system. The emergency rules are also explicit: when a grid outage is active, selected high-priority containers may be moved to a backup-powered location if temperature and power capacity allow it.

In [ ]:
class ReeferDigitalTwin:
    """Discrete-time simulation of a small reefer-terminal system."""

    def __init__(
        self,
        containers: List[ReeferContainer],
        locations: List[StorageLocation],
        assignment_costs: Dict[Tuple[int, int], float],
    ):
        self.containers = containers
        self.locations = locations
        self.assignment_costs = assignment_costs
        self._containers_by_id = {container.id: container for container in containers}
        self._locations_by_id = {location.id: location for location in locations}
        if len(self._containers_by_id) != len(containers):
            raise ValueError("Container IDs must be unique.")
        if len(self._locations_by_id) != len(locations):
            raise ValueError("Location IDs must be unique.")

        self.current_time = datetime.now()
        self.simulation_speed = 3600  # A nominal display value: 1 second per simulated hour
        self.container_assignments: Dict[int, int] = {}
        self.location_power_usage = {location.id: 0.0 for location in locations}
        self.container_temperatures = {
            container.id: container.temp_req for container in containers
        }
        self.power_grid_status = "normal"
        self.grid_outage = False

        self.sensors = self._initialize_sensors()
        self.sensor_data = deque(maxlen=1000)
        self.predictive_models = self._initialize_predictive_models()
        self.alerts: List[Dict[str, Any]] = []
        self.kpi_history = deque(maxlen=100)
        self.emergency_protocols = self._initialize_emergency_protocols()

        self.total_cargo_value = sum(container.cargo_value for container in containers)
        self.cargo_integrity_preserved = 0.0
        self.power_capacity_utilization = 0.0

    def _initialize_sensors(self) -> List[IoT_Sensor]:
        """Create the fixed set of simulated sensors used by this example."""
        sensors = []
        for location in self.locations:
            for index in range(3):
                sensors.append(
                    IoT_Sensor(
                        id=f"TEMP_L{location.id}_{index}",
                        sensor_type="temperature",
                        location_id=location.id,
                        container_id=None,
                        accuracy=0.95,
                        sampling_rate=0.1,
                        battery_level=1.0,
                        last_maintenance=datetime.now() - timedelta(days=30),
                    )
                )
            for index in range(2):
                sensors.append(
                    IoT_Sensor(
                        id=f"POWER_L{location.id}_{index}",
                        sensor_type="power",
                        location_id=location.id,
                        container_id=None,
                        accuracy=0.98,
                        sampling_rate=1.0,
                        battery_level=1.0,
                        last_maintenance=datetime.now() - timedelta(days=15),
                    )
                )

        for container in self.containers:
            sensors.append(
                IoT_Sensor(
                    id=f"TEMP_C{container.id}",
                    sensor_type="temperature",
                    location_id=None,
                    container_id=container.id,
                    accuracy=0.97,
                    sampling_rate=0.05,
                    battery_level=0.85,
                    last_maintenance=datetime.now() - timedelta(days=7),
                )
            )
            sensors.append(
                IoT_Sensor(
                    id=f"DOOR_C{container.id}",
                    sensor_type="door",
                    location_id=None,
                    container_id=container.id,
                    accuracy=0.99,
                    sampling_rate=10.0,
                    battery_level=0.90,
                    last_maintenance=datetime.now() - timedelta(days=5),
                )
            )
        return sensors

    def _initialize_predictive_models(self) -> Dict[str, Dict[str, Any]]:
        """Describe heuristic indicators; these entries are not trained models."""
        return {
            "temperature_failure_risk": {
                "method": "normalized temperature deviation",
                "horizon_hours": 0,
            },
            "power_failure_risk": {
                "method": "aggregate load threshold",
                "horizon_hours": 0,
            },
            "cargo_degradation_risk": {
                "method": "temperature deviation scaled by cargo value",
                "horizon_hours": 0,
            },
        }

    def _initialize_emergency_protocols(self) -> Dict[str, Dict[str, Any]]:
        """Set the transparent rules used by the outage response example."""
        return {
            "power_failure": {
                "priority_containers": lambda container: container.cargo_type
                in ["Pharmaceutical", "Vaccines"],
                "backup_power_locations": [
                    location.id for location in self.locations if location.backup_power
                ],
            },
            "temperature_violation": {
                "critical_containers": lambda container: container.temp_tolerance <= 1.0,
                "relocation_priority": lambda container: container.cargo_value,
            },
        }

    def assign_containers(self, assignment: Dict[int, int]) -> None:
        """Load a complete container-to-location assignment into the simulation."""
        expected_ids = set(self._containers_by_id)
        if set(assignment) != expected_ids:
            raise ValueError("The assignment must contain every container exactly once.")
        unknown_locations = set(assignment.values()) - set(self._locations_by_id)
        if unknown_locations:
            raise ValueError(f"Unknown location IDs: {sorted(unknown_locations)}")

        self.container_assignments = dict(assignment)
        self.location_power_usage = {location.id: 0.0 for location in self.locations}
        for container_id, location_id in assignment.items():
            container = self._containers_by_id[container_id]
            location = self._locations_by_id[location_id]
            self.location_power_usage[location_id] += container.energy
            self.container_temperatures[container_id] = (
                location.temp_min + location.temp_max
            ) / 2

    def simulate_step(self, duration_hours: float = 1.0) -> Dict[str, Any]:
        """Record a step, advance the physical state, and calculate diagnostics."""
        if duration_hours <= 0:
            raise ValueError("The simulation duration must be positive.")
        if not self.container_assignments:
            raise ValueError("Assign containers before running the simulation.")

        step_time = self.current_time
        sensor_readings = self._simulate_sensor_readings(step_time)
        for reading in sensor_readings:
            self.sensor_data.append(reading)

        self.current_time += timedelta(hours=duration_hours)
        self._update_container_temperatures(duration_hours)
        self._update_power_grid_status()
        step_alerts = self._generate_alerts()
        self.alerts.extend(step_alerts)
        kpi = self._calculate_kpi()
        self.kpi_history.append(kpi)
        predictions = self._generate_predictions()

        return {
            "timestamp": step_time,
            "duration": duration_hours,
            "sensor_readings": sensor_readings,
            "alerts": step_alerts,
            "kpi": kpi,
            "predictions": predictions,
        }

    def _simulate_sensor_readings(
        self, timestamp: datetime
    ) -> List[Dict[str, Any]]:
        """Generate noisy observations of the current simulated state."""
        readings = []
        for sensor in self.sensors:
            reading = {
                "sensor_id": sensor.id,
                "sensor_type": sensor.sensor_type,
                "timestamp": timestamp,
                "location_id": sensor.location_id,
                "container_id": sensor.container_id,
            }
            noise_multiplier = 1 + 5 * (1 - sensor.accuracy)

            if sensor.sensor_type == "temperature":
                if sensor.container_id is not None:
                    actual_value = self.container_temperatures[sensor.container_id]
                    noise = np.random.normal(0, 0.5 * noise_multiplier)
                else:
                    location = self._locations_by_id[sensor.location_id]
                    actual_value = (location.temp_min + location.temp_max) / 2
                    noise = np.random.normal(0, 0.3 * noise_multiplier)
                reading["value"] = actual_value + noise
                reading["unit"] = "°C"
            elif sensor.sensor_type == "power":
                actual_value = self.location_power_usage[sensor.location_id]
                noise = np.random.normal(0, 1.0 * noise_multiplier)
                reading["value"] = max(0.0, actual_value + noise)
                reading["unit"] = "kW"
            else:
                reading["value"] = int(np.random.random() < 0.05)
                reading["unit"] = "binary"

            reading["battery_level"] = sensor.battery_level
            reading["accuracy"] = sensor.accuracy
            reading["signal_strength"] = float(np.random.uniform(0.7, 1.0))
            readings.append(reading)
        return readings

    def _update_container_temperatures(self, duration_hours: float) -> None:
        """Apply the discrete-time first-order temperature model."""
        for container_id, location_id in self.container_assignments.items():
            container = self._containers_by_id[container_id]
            location = self._locations_by_id[location_id]
            current_temp = self.container_temperatures[container_id]
            target_temp = (location.temp_min + location.temp_max) / 2
            thermal_coefficient = 0.1 * container.thermal_mass * container.insulation_quality
            deterministic_change = (
                thermal_coefficient * (target_temp - current_temp) * duration_hours
            )
            fluctuation = np.random.normal(0, 0.1) * np.sqrt(duration_hours)
            self.container_temperatures[container_id] = (
                current_temp + deterministic_change + fluctuation
            )

    def _update_power_grid_status(self) -> None:
        """Classify aggregate grid loading and preserve an active outage state."""
        if self.grid_outage:
            self.power_grid_status = "failure"
            return

        total_load = sum(self.location_power_usage.values())
        total_capacity = sum(location.power_cap for location in self.locations)
        load_factor = total_load / total_capacity if total_capacity > 0 else 0.0
        weather_factor = np.random.uniform(0.9, 1.1)
        effective_load_factor = load_factor * weather_factor

        if effective_load_factor < 0.7:
            self.power_grid_status = "normal"
        elif effective_load_factor < 0.85:
            self.power_grid_status = "warning"
        elif effective_load_factor < 0.95:
            self.power_grid_status = "critical"
        else:
            self.power_grid_status = "failure"

    def _generate_alerts(self) -> List[Dict[str, Any]]:
        """Report temperature deviations, local overloads, and grid stress."""
        alerts = []
        for container_id, location_id in self.container_assignments.items():
            container = self._containers_by_id[container_id]
            current_temp = self.container_temperatures[container_id]
            deviation = abs(current_temp - container.temp_req)
            if deviation > container.temp_tolerance:
                alerts.append(
                    {
                        "type": "temperature_violation",
                        "severity": "high"
                        if deviation > 2 * container.temp_tolerance
                        else "medium",
                        "container_id": container_id,
                        "location_id": location_id,
                        "current_temp": current_temp,
                        "required_temp": container.temp_req,
                        "deviation": deviation,
                        "timestamp": self.current_time,
                    }
                )

        for location in self.locations:
            usage = self.location_power_usage[location.id]
            if usage > location.power_cap:
                alerts.append(
                    {
                        "type": "power_overload",
                        "severity": "critical"
                        if usage > 1.2 * location.power_cap
                        else "high",
                        "location_id": location.id,
                        "current_usage": usage,
                        "capacity": location.power_cap,
                        "overload": usage - location.power_cap,
                        "timestamp": self.current_time,
                    }
                )

        if self.power_grid_status in ["critical", "failure"]:
            alerts.append(
                {
                    "type": "power_grid",
                    "severity": "critical"
                    if self.power_grid_status == "failure"
                    else "high",
                    "status": self.power_grid_status,
                    "timestamp": self.current_time,
                }
            )
        return alerts

    def _calculate_kpi(self) -> Dict[str, float]:
        """Calculate temperature integrity and power-capacity indicators."""
        assigned_count = len(self.container_assignments)
        within_tolerance = sum(
            abs(self.container_temperatures[container_id] - self._containers_by_id[container_id].temp_req)
            <= self._containers_by_id[container_id].temp_tolerance
            for container_id in self.container_assignments
        )
        self.cargo_integrity_preserved = (
            100 * within_tolerance / assigned_count if assigned_count else 0.0
        )

        total_load = sum(self.location_power_usage.values())
        total_capacity = sum(location.power_cap for location in self.locations)
        self.power_capacity_utilization = (
            100 * total_load / total_capacity if total_capacity else 0.0
        )
        average_location_utilization = (
            np.mean(
                [
                    100 * self.location_power_usage[location.id] / location.power_cap
                    for location in self.locations
                    if location.power_cap > 0
                ]
            )
            if self.locations
            else 0.0
        )
        recent_alerts = sum(
            alert["timestamp"] > self.current_time - timedelta(hours=1)
            for alert in self.alerts
        )
        return {
            "cargo_integrity": self.cargo_integrity_preserved,
            "power_capacity_utilization": self.power_capacity_utilization,
            "average_location_utilization": float(average_location_utilization),
            "alert_rate": float(recent_alerts),
            "containers_assigned": float(assigned_count),
            "total_containers": float(len(self.containers)),
        }

    def _generate_predictions(self) -> Dict[str, float]:
        """Calculate heuristic risk indicators, not forecast probabilities."""
        assigned_count = len(self.container_assignments)
        if assigned_count == 0:
            return {
                "temperature_failure_risk": 0.0,
                "power_failure_risk": 0.0,
                "cargo_degradation_risk": 0.0,
            }

        normalized_deviations = []
        degradation_terms = []
        for container_id in self.container_assignments:
            container = self._containers_by_id[container_id]
            deviation = abs(self.container_temperatures[container_id] - container.temp_req)
            normalized_deviations.append(min(1.0, deviation / container.temp_tolerance))
            value_factor = container.cargo_value / 1000
            degradation_terms.append(
                min(1.0, deviation / (2 * container.temp_tolerance)) * value_factor
            )

        total_load = sum(self.location_power_usage.values())
        total_capacity = sum(location.power_cap for location in self.locations)
        load_factor = total_load / total_capacity if total_capacity else 0.0
        return {
            "temperature_failure_risk": 100 * float(np.mean(normalized_deviations)),
            "power_failure_risk": max(0.0, (load_factor - 0.8) * 100),
            "cargo_degradation_risk": 100 * float(np.mean(degradation_terms)),
        }

    def run_scenario(
        self, scenario_name: str, duration_hours: float = 24.0
    ) -> Dict[str, Any]:
        """Run normal, grid-outage, or sensor-degradation simulation steps."""
        if duration_hours <= 0:
            raise ValueError("The scenario duration must be positive.")
        if scenario_name not in {
            "normal_operation",
            "hurricane_patricia",
            "sensor_failure",
        }:
            raise ValueError(f"Unknown scenario: {scenario_name}")

        scenario = {
            "scenario": scenario_name,
            "duration": duration_hours,
            "steps": [],
            "summary": {},
            "alerts": [],
            "kpi_trend": [],
        }
        for hour in range(int(duration_hours)):
            if scenario_name == "hurricane_patricia" and hour == 6:
                self.grid_outage = True
                self.power_grid_status = "failure"

            if scenario_name == "sensor_failure" and hour == 8:
                failed_count = int(len(self.sensors) * 0.2)
                for sensor in random.sample(self.sensors, failed_count):
                    sensor.accuracy *= 0.5
                    sensor.battery_level *= 0.7

            step = self.simulate_step(1.0)
            if scenario_name == "hurricane_patricia" and hour == 7:
                step["emergency_actions"] = self._execute_emergency_protocols()
            scenario["steps"].append(step)
            scenario["kpi_trend"].append(step["kpi"])
            scenario["alerts"].extend(step["alerts"])

        if scenario["kpi_trend"]:
            first_kpi = scenario["kpi_trend"][0]
            final_kpi = scenario["kpi_trend"][-1]
            scenario["summary"] = {
                "initial_cargo_integrity": first_kpi["cargo_integrity"],
                "final_cargo_integrity": final_kpi["cargo_integrity"],
                "integrity_preservation": final_kpi["cargo_integrity"],
                "final_power_capacity_utilization": final_kpi[
                    "power_capacity_utilization"
                ],
                "total_alerts": len(scenario["alerts"]),
                "high_severity_alerts": sum(
                    alert["severity"] in ["high", "critical"]
                    for alert in scenario["alerts"]
                ),
            }
        return scenario

    def _execute_emergency_protocols(self) -> Dict[str, Any]:
        """Move eligible priority containers when a backup location can accept them."""
        actions = {
            "timestamp": self.current_time,
            "protocols_executed": [],
            "containers_moved": [],
            "backup_power_activated": [],
            "critical_containers_protected": 0,
        }
        if self.power_grid_status != "failure":
            return actions

        protocol = self.emergency_protocols["power_failure"]
        actions["protocols_executed"].append("power_failure")
        backup_ids = protocol["backup_power_locations"]
        actions["backup_power_activated"] = list(backup_ids)
        priority_containers = [
            container
            for container in self.containers
            if protocol["priority_containers"](container)
        ]

        for container in priority_containers:
            current_location_id = self.container_assignments[container.id]
            if current_location_id in backup_ids:
                continue
            for backup_id in backup_ids:
                backup_location = self._locations_by_id[backup_id]
                fits_power = (
                    self.location_power_usage[backup_id] + container.energy
                    <= backup_location.power_cap
                )
                fits_temperature = (
                    backup_location.temp_min
                    <= container.temp_req
                    <= backup_location.temp_max
                )
                if fits_power and fits_temperature:
                    self.container_assignments[container.id] = backup_id
                    self.location_power_usage[current_location_id] -= container.energy
                    self.location_power_usage[backup_id] += container.energy
                    actions["containers_moved"].append(
                        {
                            "container_id": container.id,
                            "from_location": current_location_id,
                            "to_location": backup_id,
                            "reason": "priority_backup_power",
                        }
                    )
                    break

        actions["critical_containers_protected"] = len(actions["containers_moved"])
        return actions

print("The digital-twin simulation class is ready.")

## 3. Load an illustrative assignment

This assignment is a fixed input for observing the simulation; it is not the output of an optimization procedure. `assign_containers` checks that every container has exactly one known location, then calculates location loads and initializes each container at the midpoint of its assigned location's temperature range. The resulting loads are checked and reported during simulation, not enforced as assignment constraints.

In [ ]:
digital_twin = ReeferDigitalTwin(containers, locations, assignment_costs)

sample_assignment = {
    1: 1,
    2: 3,
    3: 1,
    4: 4,
    5: 2,
    6: 3,
    7: 1,
    8: 2,
}
digital_twin.assign_containers(sample_assignment)

print("Digital twin initialized:")
print(f"  Containers: {len(containers)}")
print(f"  Locations: {len(locations)}")
print(f"  Simulated sensors: {len(digital_twin.sensors)}")
print(f"  Total cargo value: ${digital_twin.total_cargo_value:.0f}K")

print("\nIllustrative assignment:")
for container_id, location_id in sample_assignment.items():
    container = digital_twin._containers_by_id[container_id]
    location = digital_twin._locations_by_id[location_id]
    print(
        f"  C{container_id} ({container.cargo_type}) -> "
        f"L{location_id} ({location.location_type})"
    )

print("\nInitial power demand by location:")
for location in locations:
    usage = digital_twin.location_power_usage[location.id]
    utilization = 100 * usage / location.power_cap
    status = "OVER CAPACITY" if usage > location.power_cap else "within capacity"
    print(
        f"  L{location.id}: {usage:.1f}/{location.power_cap} kW "
        f"({utilization:.1f}%; {status})"
    )

sensor_counts = {}
for sensor in digital_twin.sensors:
    sensor_counts[sensor.sensor_type] = sensor_counts.get(sensor.sensor_type, 0) + 1
print("\nSimulated sensor configuration:")
for sensor_type, count in sorted(sensor_counts.items()):
    print(f"  {sensor_type.capitalize()}: {count}")

print("\nHeuristic indicators:")
for name, details in digital_twin.predictive_models.items():
    print(f"  {name.replace('_', ' ').title()}: {details['method']}")

## 4. Run normal-operation simulation

Each step produces one reading per simulated sensor. The sensor readings are a snapshot taken at the beginning of the step; temperatures and alerts are then calculated after the state advances. Random noise means numerical readings can differ between runs. No random seed is specified, so exact simulation values are not intended to be reproducible.

In [ ]:
print("=" * 60)
print("DIGITAL TWIN: NORMAL-OPERATION SCENARIO")
print("=" * 60)

normal_scenario = digital_twin.run_scenario("normal_operation", duration_hours=12)
summary = normal_scenario["summary"]
print("\nNormal-operation summary:")
print(f"  Final cargo integrity: {summary['final_cargo_integrity']:.1f}%")
print(
    "  Aggregate power-capacity utilization: "
    f"{summary['final_power_capacity_utilization']:.1f}%"
)
print(f"  Alerts recorded: {summary['total_alerts']}")
print(f"  High-severity alerts: {summary['high_severity_alerts']}")

all_readings = [
    reading
    for step in normal_scenario["steps"]
    for reading in step["sensor_readings"]
]
reading_groups = {}
for reading in all_readings:
    group = reading_groups.setdefault(reading["sensor_type"], [])
    group.append(reading["value"])

print("\nSensor-reading summaries:")
for sensor_type, values in sorted(reading_groups.items()):
    print(
        f"  {sensor_type.capitalize()}: {len(values)} readings, "
        f"mean={np.mean(values):.2f}, standard deviation={np.std(values):.2f}"
    )

kpi_trend = normal_scenario["kpi_trend"]
if len(kpi_trend) > 1:
    initial_integrity = kpi_trend[0]["cargo_integrity"]
    final_integrity = kpi_trend[-1]["cargo_integrity"]
    initial_utilization = kpi_trend[0]["power_capacity_utilization"]
    final_utilization = kpi_trend[-1]["power_capacity_utilization"]
    print("\nKPI trend:")
    print(
        f"  Cargo integrity: {initial_integrity:.1f}% -> {final_integrity:.1f}% "
        f"({final_integrity - initial_integrity:+.1f} percentage points)"
    )
    print(
        f"  Power-capacity utilization: {initial_utilization:.1f}% -> "
        f"{final_utilization:.1f}% "
        f"({final_utilization - initial_utilization:+.1f} percentage points)"
    )
    print(
        "  Mean location utilization: "
        f"{np.mean([kpi['average_location_utilization'] for kpi in kpi_trend]):.1f}%"
    )
    print(
        "  Mean recent-alert count per step: "
        f"{np.mean([kpi['alert_rate'] for kpi in kpi_trend]):.1f}"
    )

alert_counts = {}
for alert in normal_scenario["alerts"]:
    key = (alert["type"], alert["severity"])
    alert_counts[key] = alert_counts.get(key, 0) + 1
print("\nAlerts by type and severity:")
if alert_counts:
    for (alert_type, severity), count in sorted(alert_counts.items()):
        print(f"  {alert_type.replace('_', ' ').title()} ({severity}): {count}")
else:
    print("  No alerts were generated during this run.")

## 5. Examine a grid-outage scenario

The named hurricane scenario applies a simulated grid outage at hour 6 and invokes the emergency rules at hour 7. The event is a scenario assumption, not a reconstruction of a historical hurricane. The model reports the effects represented by its rules; it does not establish the real-world effectiveness of a digital twin or emergency response.

In [ ]:
digital_twin_hurricane = ReeferDigitalTwin(containers, locations, assignment_costs)
digital_twin_hurricane.assign_containers(sample_assignment)
hurricane_scenario = digital_twin_hurricane.run_scenario(
    "hurricane_patricia", duration_hours=24
)
summary = hurricane_scenario["summary"]

print("=" * 60)
print("DIGITAL TWIN: GRID-OUTAGE SCENARIO")
print("=" * 60)
print(f"\nFinal cargo integrity: {summary['final_cargo_integrity']:.1f}%")
print(f"Initial cargo integrity: {summary['initial_cargo_integrity']:.1f}%")
print(
    "Final aggregate power-capacity utilization: "
    f"{summary['final_power_capacity_utilization']:.1f}%"
)
print(f"Alerts recorded: {summary['total_alerts']}")
print(f"High-severity alerts: {summary['high_severity_alerts']}")

action_steps = [
    step for step in hurricane_scenario["steps"] if "emergency_actions" in step
]
if action_steps:
    actions = action_steps[0]["emergency_actions"]
    print("\nEmergency-rule summary:")
    print(f"  Protocols executed: {actions['protocols_executed']}")
    print(f"  Containers moved: {actions['critical_containers_protected']}")
    print(f"  Backup locations considered: {actions['backup_power_activated']}")
    for move in actions["containers_moved"]:
        container = digital_twin_hurricane._containers_by_id[move["container_id"]]
        print(
            f"  C{move['container_id']} ({container.cargo_type}): "
            f"L{move['from_location']} -> L{move['to_location']}"
        )

first_timestamp = hurricane_scenario["steps"][0]["timestamp"]
alert_timeline = {}
for alert in hurricane_scenario["alerts"]:
    hour = int((alert["timestamp"] - first_timestamp).total_seconds() / 3600)
    alert_timeline[hour] = alert_timeline.get(hour, 0) + 1
print("\nAlert counts at four-hour intervals:")
for hour in range(0, 25, 4):
    print(f"  Hour {hour:2d}: {alert_timeline.get(hour, 0)} alerts")

risk_history = [step["predictions"] for step in hurricane_scenario["steps"]]
if risk_history:
    print("\nMean heuristic risk indicators (not calibrated probabilities):")
    for key in risk_history[0]:
        mean_risk = np.mean([record[key] for record in risk_history])
        print(f"  {key.replace('_', ' ').title()}: {mean_risk:.1f} index points")

## 6. Visualize the simulation

The dashboard summarizes the normal-operation run and the assignment loaded into `digital_twin`. Temperature colors indicate whether each container is within its stated tolerance. Power bars are compared with location capacities. Sensor types are shown separately because their readings have different units; combining them in one histogram would be misleading.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Reefer-Terminal Digital-Twin Simulation", fontsize=16, weight="bold")

# Sensor distributions are separated by type because their units differ.
ax = axes[0, 0]
for sensor_type, values in sorted(reading_groups.items()):
    ax.hist(values, bins=15, alpha=0.6, label=sensor_type.capitalize())
ax.set_xlabel("Reading value (units depend on sensor type)")
ax.set_ylabel("Frequency")
ax.set_title("Simulated sensor readings")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
risk_keys = [
    "temperature_failure_risk",
    "power_failure_risk",
    "cargo_degradation_risk",
]
for key in risk_keys:
    values = [step["predictions"][key] for step in normal_scenario["steps"]]
    ax.plot(range(len(values)), values, label=key.replace("_", " ").title())
ax.set_xlabel("Simulation step (hours)")
ax.set_ylabel("Heuristic index (not a probability)")
ax.set_title("Heuristic risk indicators")
ax.set_ylim(bottom=0)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[0, 2]
normal_kpis = normal_scenario["kpi_trend"]
steps = range(len(normal_kpis))
ax.plot(steps, [item["cargo_integrity"] for item in normal_kpis], label="Cargo integrity")
ax.plot(
    steps,
    [item["power_capacity_utilization"] for item in normal_kpis],
    label="Aggregate power-capacity utilization",
)
ax.plot(
    steps,
    [item["average_location_utilization"] for item in normal_kpis],
    label="Mean location utilization",
)
ax.set_xlabel("Simulation step (hours)")
ax.set_ylabel("Percentage (%)")
ax.set_title("KPI trends")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
container_ids = list(digital_twin.container_temperatures)
temperatures = [digital_twin.container_temperatures[cid] for cid in container_ids]
colors = []
for container_id, temperature in zip(container_ids, temperatures):
    container = digital_twin._containers_by_id[container_id]
    colors.append(
        "tab:green"
        if abs(temperature - container.temp_req) <= container.temp_tolerance
        else "tab:red"
    )
ax.bar([f"C{cid}" for cid in container_ids], temperatures, color=colors, alpha=0.75)
ax.set_xlabel("Container")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Container temperatures after normal scenario")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color="tab:green", label="Within tolerance"),
        plt.Rectangle((0, 0), 1, 1, color="tab:red", label="Outside tolerance"),
    ]
)

ax = axes[1, 1]
location_ids = [location.id for location in locations]
loads = [digital_twin.location_power_usage[location_id] for location_id in location_ids]
capacities = [digital_twin._locations_by_id[location_id].power_cap for location_id in location_ids]
positions = np.arange(len(location_ids))
ax.bar(positions, loads, alpha=0.75, label="Assigned load")
ax.plot(positions, capacities, "r--o", label="Power capacity")
ax.set_xticks(positions)
ax.set_xticklabels([f"L{location_id}" for location_id in location_ids])
ax.set_xlabel("Location")
ax.set_ylabel("Power (kW)")
ax.set_title("Location power load and capacity")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
for position, load, capacity in zip(positions, loads, capacities):
    ax.text(position, load + 2, f"{100 * load / capacity:.0f}%", ha="center")

ax = axes[1, 2]
alert_severities = ["medium", "high", "critical"]
alert_types = sorted({alert["type"] for alert in normal_scenario["alerts"]})
if alert_types:
    bottoms = np.zeros(len(alert_types))
    for severity in alert_severities:
        counts = [
            sum(
                alert["type"] == alert_type and alert["severity"] == severity
                for alert in normal_scenario["alerts"]
            )
            for alert_type in alert_types
        ]
        ax.bar(alert_types, counts, bottom=bottoms, label=severity.title())
        bottoms += np.asarray(counts)
    ax.set_xticks(range(len(alert_types)))
    ax.set_xticklabels(
        [alert_type.replace("_", " ").title() for alert_type in alert_types],
        rotation=35,
        ha="right",
    )
    ax.legend(title="Severity")
else:
    ax.text(0.5, 0.5, "No alerts in this run", ha="center", va="center", transform=ax.transAxes)
ax.set_xlabel("Alert type")
ax.set_ylabel("Number of alerts")
ax.set_title("Alerts by type and severity")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Examine simulation scaling

This experiment repeats the demonstration container and location records to create larger **synthetic** instances. New IDs are assigned to each copy so that dictionary-based state remains unambiguous. The experiment measures runtime and the number of generated readings; it does not establish a general complexity bound or prove linear scaling. Runtime depends on the machine, software environment, and instance details.

In [ ]:
def simulate_scalability(
    num_containers: int,
    num_locations: int,
    duration_hours: int = 6,
) -> Dict[str, float]:
    """Run the same step-by-step simulation on a larger synthetic instance."""
    if num_containers <= 0 or num_locations <= 0:
        raise ValueError("Scale sizes must be positive.")

    scaled_containers = [
        replace(containers[index % len(containers)], id=index + 1)
        for index in range(num_containers)
    ]
    scaled_locations = [
        replace(locations[index % len(locations)], id=index + 1)
        for index in range(num_locations)
    ]
    scaled_costs = {
        (container.id, location.id): assignment_costs[
            (
                containers[container_index % len(containers)].id,
                locations[location_index % len(locations)].id,
            )
        ]
        for container_index, container in enumerate(scaled_containers)
        for location_index, location in enumerate(scaled_locations)
    }

    scaled_twin = ReeferDigitalTwin(scaled_containers, scaled_locations, scaled_costs)
    scaled_assignment = {
        container.id: scaled_locations[index % num_locations].id
        for index, container in enumerate(scaled_containers)
    }
    scaled_twin.assign_containers(scaled_assignment)

    start_time = time.perf_counter()
    scenario = scaled_twin.run_scenario("normal_operation", duration_hours)
    elapsed_seconds = time.perf_counter() - start_time
    reading_count = sum(
        len(step["sensor_readings"]) for step in scenario["steps"]
    )
    return {
        "num_containers": num_containers,
        "num_locations": num_locations,
        "num_sensors": len(scaled_twin.sensors),
        "simulation_time": elapsed_seconds,
        "data_points": reading_count,
        "final_integrity": scenario["summary"]["final_cargo_integrity"],
        "total_alerts": scenario["summary"]["total_alerts"],
    }

scale_tests = [(8, 4), (20, 8), (50, 15), (100, 25)]
scale_results = []
for container_count, location_count in scale_tests:
    result = simulate_scalability(container_count, location_count)
    scale_results.append(result)
    throughput = result["data_points"] / result["simulation_time"]
    print(
        f"{container_count} containers, {location_count} locations: "
        f"{result['simulation_time']:.4f} seconds, "
        f"{result['data_points']:,} readings, "
        f"{throughput:.0f} readings/second"
    )

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Synthetic Simulation-Scaling Experiment", fontsize=15, weight="bold")
container_counts = [result["num_containers"] for result in scale_results]
simulation_times = [result["simulation_time"] for result in scale_results]
data_points = [result["data_points"] for result in scale_results]
sensor_counts = [result["num_sensors"] for result in scale_results]
final_integrities = [result["final_integrity"] for result in scale_results]
alert_counts = [result["total_alerts"] for result in scale_results]

axes[0, 0].plot(container_counts, simulation_times, "o-")
axes[0, 0].set_xlabel("Number of containers")
axes[0, 0].set_ylabel("Runtime (seconds)")
axes[0, 0].set_title("Runtime by instance size")
axes[0, 0].grid(True, alpha=0.3)

throughput = [points / elapsed for points, elapsed in zip(data_points, simulation_times)]
axes[0, 1].plot(container_counts, throughput, "s-", color="tab:green")
axes[0, 1].set_xlabel("Number of containers")
axes[0, 1].set_ylabel("Generated readings per second")
axes[0, 1].set_title("Observed reading throughput")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(container_counts, sensor_counts, "^-", color="tab:red")
axes[1, 0].set_xlabel("Number of containers")
axes[1, 0].set_ylabel("Number of simulated sensors")
axes[1, 0].set_title("Sensor count by instance size")
axes[1, 0].grid(True, alpha=0.3)

positions = np.arange(len(container_counts))
axes[1, 1].bar(positions - 0.2, final_integrities, width=0.4, label="Cargo integrity (%)")
axes[1, 1].bar(positions + 0.2, alert_counts, width=0.4, label="Alert count")
axes[1, 1].set_xticks(positions)
axes[1, 1].set_xticklabels([str(count) for count in container_counts])
axes[1, 1].set_xlabel("Number of containers")
axes[1, 1].set_ylabel("Metric value")
axes[1, 1].set_title("Run outcomes by instance size")
axes[1, 1].legend()
axes[1, 1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Interpretation and practical use

### What the simulation can show

- How a supplied assignment distributes power demand across locations.
- Whether modeled location loads exceed their stated capacities.
- How the assumed location temperature targets and thermal coefficients affect container temperatures.
- How the chosen alert rules respond to temperature deviations, overloads, and grid loading.
- How the number of simulated sensors and generated readings changes in the synthetic scaling experiment.

### What the results cannot establish

This small simulation does not demonstrate real-time synchronization, validated prediction accuracy, cargo-integrity outcomes in a historical emergency, or the economic value of a digital-twin deployment. Its random sensor noise and simplified physical rules are modeling choices, not measured terminal behavior. The scaling experiment reports measurements from the current run and environment; it does not justify a universal statement about computational complexity.

### Relationship to the SLAP

A storage location assignment problem requires a decision model that selects container locations under stated objectives and constraints. This notebook instead accepts an assignment and simulates selected consequences. Assignment costs are present in the input but are not optimized, and location power capacity is monitored rather than imposed as a feasibility constraint. The simulation can therefore complement an assignment method by examining a proposed plan, but its outputs should not be mistaken for an optimal SLAP solution.

A production digital twin would require validated physical models, calibrated sensors, reliable data exchange, uncertainty analysis, security controls, operational constraints, and verification against observed terminal behavior. Those capabilities are outside this illustrative simulation.